# 03c — Filtro de Liquidez (Pós-Integridade)

**Parte 3 de 3 do passo 3 do pipeline.** Lê a lista de exclusão gerada pelo Classificador de
Integridade (`data/Tickers/tickers_excluidos_integridade.csv`), aplica sobre o universo pós-ADTV
e grava a **matriz de preços final** (`Matriz_precos_sanitizada.csv`) e os artefatos de auditoria.

**Pré-requisitos:**
- `03_01a_Pre_Integridade.ipynb` já rodou → `data/Tickers/universo_pos_liquidez.csv` existe
- `Apendice_Classificador_Integridade_Universo_v2.ipynb` já rodou → `data/Tickers/tickers_excluidos_integridade.csv` existe

**Saídas deste notebook:**
- `data/Matriz_Precos/Matriz_precos_sanitizada.csv` — artefato autoritativo consumido pelo NB05
- `data/Tickers/tickers_finais.csv` — lista final de tickers
- Log de auditoria de exclusões

In [1]:
import sys
import time
from pathlib import Path
import logging
import numpy as np
import pandas as pd

from utils.filtros import filtrar_presenca, filtrar_integridade_ipo, filtrar_adtv_formacao, executar_testes_integridade
from utils.auditoria import gerar_auditoria_exclusoes
from utils.config_loader import carregar_parametros

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] - %(message)s")

print("=== 03c — Filtro Pós-Integridade ===")
print(f"Python: {sys.version}")
print(f"Pandas: {pd.__version__} | NumPy: {np.__version__}")

=== 03c — Filtro Pós-Integridade ===
Python: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
Pandas: 3.0.3 | NumPy: 2.4.6


In [2]:
# ── Caminhos ──────────────────────────────────────────────────────────────────
PARENT_PATH       = Path.cwd().parent.parent
DIRETORIO_CACHE   = PARENT_PATH / "data" / "dados_economatica_consolidados"
DIRETORIO_DESTINO = PARENT_PATH / "data" / "Matriz_Precos"
DIR_TICKERS       = PARENT_PATH / "data" / "Tickers"

DIRETORIO_DESTINO.mkdir(parents=True, exist_ok=True)
DIR_TICKERS.mkdir(parents=True, exist_ok=True)

# ── Parâmetros ────────────────────────────────────────────────────────────────
config             = carregar_parametros()
DATA_INICIO        = pd.to_datetime(config["DATA_INICIO"])
DATA_FIM           = pd.to_datetime(config["DATA_FIM"])
LIMIAR_PRESENCA    = config["LIMIAR_PRESENCA"]
ANO_FORMACAO_ADTV  = config["ANO_FORMACAO_ADTV"]
PERCENTIL_LIQUIDEZ = config["PERCENTIL_LIQUIDEZ"]

# ── Verifica pré-requisitos ───────────────────────────────────────────────────
arq_universo = DIR_TICKERS / "universo_pos_liquidez.csv"
arq_excl     = DIR_TICKERS / "tickers_excluidos_integridade.csv"

if not arq_universo.exists():
    raise FileNotFoundError(
        f"[ERRO] universo_pos_liquidez.csv não encontrado em {DIR_TICKERS}.\n"
        "Execute primeiro: 03_01a_Pre_Integridade.ipynb"
    )
if not arq_excl.exists():
    raise FileNotFoundError(
        f"[ERRO] tickers_excluidos_integridade.csv não encontrado em {DIR_TICKERS}.\n"
        "Execute primeiro: Apendice_Classificador_Integridade_Universo_v2.ipynb\n"
        f"  com PASTA_SAIDA apontando para {DIR_TICKERS}"
    )

print(f"[OK] Pré-requisitos encontrados em {DIR_TICKERS}")
print(f"     universo_pos_liquidez.csv: {arq_universo.stat().st_size} bytes")
print(f"     tickers_excluidos_integridade.csv: {arq_excl.stat().st_size} bytes")

[OK] Pré-requisitos encontrados em C:\VSCodeWorkspace\1_TCC_Final\data\Tickers
     universo_pos_liquidez.csv: 836 bytes
     tickers_excluidos_integridade.csv: 1955 bytes


In [3]:
# ── Re-executa os filtros de liquidez para reconstruir df_pos_liquidez ────────
#
# Motivo: o 03a e o 03c correm em processos separados (nbconvert independentes).
# Não é possível compartilhar estado em memória entre notebooks. Refazer os
# filtros é rápido (<2s via parquet) e garante que o estado é idêntico ao 03a.
#
caminho_parquet = DIRETORIO_CACHE / "dados_brutos_economatica.parquet"
caminho_csv     = DIRETORIO_CACHE / "dados_brutos_economatica.csv"

t0 = time.perf_counter()
if caminho_parquet.exists():
    df_consolidado = pd.read_parquet(caminho_parquet)
elif caminho_csv.exists():
    df_consolidado = pd.read_csv(
        caminho_csv, sep=";", header=[0, 1], index_col=0, parse_dates=True
    )
else:
    raise FileNotFoundError(f"Nenhum arquivo consolidado em {DIRETORIO_CACHE}")

df_precos_brutos  = df_consolidado["Fechamento"]
df_volumes_brutos = df_consolidado["Volume$"]

df_precos  = df_precos_brutos.loc[DATA_INICIO:DATA_FIM]
df_volumes = df_volumes_brutos.loc[DATA_INICIO:DATA_FIM]

df_calendario = df_precos.dropna(how="all")
df_volumes    = df_volumes.reindex(df_calendario.index)

from utils.filtros import filtrar_presenca, filtrar_integridade_ipo, filtrar_adtv_formacao

proporcao_presenca, aprovados_presenca, reprovados_presenca = filtrar_presenca(
    df_calendario, LIMIAR_PRESENCA
)
df_liquidos = df_calendario[aprovados_presenca].copy()

data_inicio_janela, ativos_integros, ativos_ipo_tardio = filtrar_integridade_ipo(df_liquidos)
df_integros = df_liquidos[ativos_integros].copy()

adtv, limiar_adtv, ativos_liquidos, ativos_iliquidos = filtrar_adtv_formacao(
    df_volumes, ativos_integros, ANO_FORMACAO_ADTV, PERCENTIL_LIQUIDEZ
)
df_pos_liquidez = df_integros[ativos_liquidos].copy()

t1 = time.perf_counter()
print(f"[03c] Filtros de liquidez re-executados em {t1-t0:.2f}s")
print(f"      Universo pós-ADTV: {df_pos_liquidez.shape[1]} ativos (deve bater com 03a)")

# Consistência com o 03a
universo_03a = pd.read_csv(arq_universo)["ticker"].astype(str).str.strip().tolist()
universo_03c = list(df_pos_liquidez.columns)
if set(universo_03a) != set(universo_03c):
    diff = set(universo_03a).symmetric_difference(set(universo_03c))
    raise AssertionError(
        f"[ERRO] Universo pós-ADTV diverge entre 03a e 03c: {diff}\n"
        "Verifique se o parquet de entrada mudou entre as execuções."
    )
print(f"[OK] Universo 03a == 03c ({len(universo_03c)} ativos) — consistência confirmada")

[03c] Filtros de liquidez re-executados em 0.27s
      Universo pós-ADTV: 118 ativos (deve bater com 03a)
[OK] Universo 03a == 03c (118 ativos) — consistência confirmada


In [4]:
# ── Aplica exclusão de integridade ────────────────────────────────────────────
excl = pd.read_csv(arq_excl)["ticker"].astype(str).str.strip().tolist()

no_universo   = [t for t in excl if t in df_pos_liquidez.columns]
fora_universo = [t for t in excl if t not in df_pos_liquidez.columns]

n_antes = df_pos_liquidez.shape[1]
df_pos_liquidez = df_pos_liquidez.drop(columns=no_universo)
n_depois = df_pos_liquidez.shape[1]

print(f"[Integridade] excluídos: {len(no_universo)} | {n_antes} → {n_depois} ativos")

if fora_universo:
    print(
        f"\n[AVISO] {len(fora_universo)} ticker(s) da lista NÃO estão no universo corrente "
        f"(foram excluídos em etapas anteriores): {fora_universo}\n"
        "  Isso é esperado se o Classificador processou um universo ligeiramente diferente."
    )

# Garante que nenhum excluído sobrou
assert not (set(no_universo) & set(df_pos_liquidez.columns)), \
    "[ERRO] Exclusão de integridade falhou — ticker excluído ainda presente."

# Grava a lista final de tickers
(pd.Series(list(df_pos_liquidez.columns), name="ticker")
   .to_csv(DIR_TICKERS / "tickers_finais.csv", index=False))
print(f"\n[OK] tickers_finais.csv gravado: {n_depois} ativos")

[Integridade] excluídos: 16 | 118 → 102 ativos

[OK] tickers_finais.csv gravado: 102 ativos


In [5]:
# ── Forward fill ──────────────────────────────────────────────────────────────
n_nulos_antes  = int(df_pos_liquidez.isna().sum().sum())
df_sanitizado  = df_pos_liquidez.ffill()
n_nulos_depois = int(df_sanitizado.isna().sum().sum())

print(f"[ffill] lacunas preenchidas: {n_nulos_antes - n_nulos_depois}")
print(f"Nulos residuais: {n_nulos_depois} | dimensão final: {df_sanitizado.shape}")

[ffill] lacunas preenchidas: 813
Nulos residuais: 0 | dimensão final: (3967, 102)


In [6]:
# ── Testes de integridade ─────────────────────────────────────────────────────
executar_testes_integridade(df_sanitizado, adtv, limiar_adtv)
print("[OK] Todos os testes de integridade (T1–T6) aprovados.")

[OK] Matriz sanitizada aprovada em todos os testes (T1-T6).
[OK] Todos os testes de integridade (T1–T6) aprovados.


In [7]:
# ── Persistência ──────────────────────────────────────────────────────────────
caminho_matriz = DIRETORIO_DESTINO / "Matriz_precos_sanitizada.csv"
df_sanitizado.to_csv(caminho_matriz, index=True)

gerar_auditoria_exclusoes(
    df_sanitizado=df_sanitizado,
    reprovados_presenca=reprovados_presenca,
    ativos_ipo_tardio=ativos_ipo_tardio,
    ativos_iliquidos=ativos_iliquidos,
    proporcao_presenca=proporcao_presenca,
    adtv=adtv,
    limiar_presenca=LIMIAR_PRESENCA,
    percentil_liquidez=PERCENTIL_LIQUIDEZ,
    diretorio_destino=DIRETORIO_DESTINO,
    aprovados_presenca=aprovados_presenca,
    ativos_integros=ativos_integros,
    ativos_liquidos=ativos_liquidos,
)

print("=" * 60)
print("FUNIL METODOLÓGICO COMPLETO")
print("=" * 60)
print(f"  Universo Bruto:                  {df_precos_brutos.shape[1]}")
print(f"  Após Presença (IV):              {len(aprovados_presenca)}")
print(f"  Após IPO / Integridade (V):      {len(ativos_integros)}")
print(f"  Após ADTV (VI):                  {len(ativos_liquidos)}")
print(f"  Após Integridade COTAHIST (VII): {n_depois}  ← Universo Final")
print("=" * 60)
print(f"[OK] Matriz salva: {caminho_matriz}")
print(f"     {n_depois} ativos × {df_sanitizado.shape[0]} pregões")

[OK] Matriz final salva: 102 ativos × 3967 pregões
[OK] Log de exclusões salvo contendo 378 registros.

FUNIL METODOLÓGICO DE ATIVOS
  Universo Bruto Ingerido:          496
  Após Filtro de Presença (IV):     135
  Após Filtro de Integridade (V):   131
  Após Filtro de ADTV (VI):         118  <- Universo Investível Final
FUNIL METODOLÓGICO COMPLETO
  Universo Bruto:                  496
  Após Presença (IV):              135
  Após IPO / Integridade (V):      131
  Após ADTV (VI):                  118
  Após Integridade COTAHIST (VII): 102  ← Universo Final
[OK] Matriz salva: C:\VSCodeWorkspace\1_TCC_Final\data\Matriz_Precos\Matriz_precos_sanitizada.csv
     102 ativos × 3967 pregões


# Autoavaliação — *Ten Simple Rules* (Rule et al., 2019)

> Rule A, Birmingham A, Zuniga C, Altintas I, Huang S-C, Knight R, Moshiri N, Nguyen MH,
> Rosenthal SB, Pérez F, Rose PW. *Ten simple rules for writing and sharing computational analyses
> in Jupyter Notebooks.* PLOS Comput Biol. 2019;15(7):e1007007.

| Regra | Tema | Status | Evidência / Aplicação no NB 03_01c (Pós-Integridade) |
| :---: | :--- | :---: | :--- |
| 1 | Contar uma história | ✅ | Explica a aplicação da lista de exclusão do classificador de integridade sobre a base de liquidez. |
| 2 | Documentar o processo | ✅ | Documenta a filtragem das 16 penny stocks/ativos sob distress severo na Etapa VII. |
| 3 | Divisão clara de células | ✅ | Células modulares para filtragem e gravação final dos dados. |
| 4 | Modularizar código | ✅ | Utiliza funções padrão e importador do config central. |
| 5 | Registrar dependências | ✅ | Dependências documentadas no repositório. |
| 6 | Controle de versão | ✅ | Código sob controle de versão. |
| 7 | Construir um pipeline | ✅ | Terceiro passo da Etapa 3 unificada, gerando a matriz final Matriz_precos_sanitizada.csv. |
| 8 | Compartilhar/explicar dados | ✅ | Explica a aplicação das exclusões de integridade a partir do classificador. |
| 9 | Ler, rodar e explorar | ✅ | Executável sem estados ocultos. |
| 10 | Pesquisa aberta | ✅ | Código livre sob licença aberta. |
